<a href="https://colab.research.google.com/github/elkins/synth-pdb/blob/main/examples/interactive_tutorials/protein_dynamics_theater.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The Protein Dynamics Theater: Normal Modes & Molecular Motion

Proteins are not static sculptures — they are **dynamic machines** whose function depends
on motion. The same ubiquitin that appears rigid in a crystal structure is constantly
breathing, flexing, and exploring conformational space in solution.

**Normal Mode Analysis (NMA)** is the mathematical tool that describes this motion
without running expensive molecular dynamics simulations. It diagonalizes the protein's
force-constant matrix (the Hessian) to extract the **eigenvectors** — the collective
motions the protein can perform — and **eigenvalues** — how much energy each motion costs.

The lowest-frequency modes (smallest eigenvalues) describe the largest, slowest motions:
the global "breathing" of the entire molecule. These are the motions most relevant
to binding, allostery, and conformational change.

> **In this tutorial** we build an Elastic Network Model, extract its principal motions,
> visualize the displacement patterns, animate the "breathing" trajectory, and connect
> the motions to NMR-measurable quantities like B-factors and RMSF.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q synth-pdb synth-dynamics biotite matplotlib "numpy<2" MDAnalysis
else:
    sys.path.insert(0, os.path.abspath("../../"))

import io
import tempfile
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.animation as animation
import matplotlib.cm as cm
from IPython.display import HTML
import MDAnalysis as mda  # noqa: N813
import biotite.structure.io.pdb as bpdb
from synth_pdb.generator import generate_pdb_content
from synth_dynamics import ANMForceField, LangevinIntegrator, Simulation, System

print("✅ Environment ready")

## The Elastic Network Model: Springs Between Neighbours

The **Anisotropic Network Model (ANM)** represents a protein as a set of Cα beads
connected by harmonic springs whenever two residues are within a cutoff distance (typically
12–15 Å). Despite this dramatic simplification, ANM captures the **functional motions** of
proteins with remarkable fidelity — motions that are dominated by topology, not by the
details of atomic interactions.

The force constant matrix (Hessian **H**) is assembled from pairwise spring interactions:

$$H_{ij} = -\frac{\partial^2 V}{\partial x_i \partial x_j} = -k \cdot \mathbf{\hat{r}}_{ij} \otimes \mathbf{\hat{r}}_{ij}$$

Diagonalising **H** gives 3N eigenvalues (frequencies²) and 3N eigenvectors (modes).
The first 6 modes are zero (rigid-body translation/rotation). Mode 7 is the **slowest
internal motion** — the global functional mode.

In [ ]:
# ── Generate ubiquitin ──
UBQ_SEQ = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"
print("Generating ubiquitin (76 residues)...")
pdb_str = generate_pdb_content(sequence_str=UBQ_SEQ, conformation="alpha", seed=42)
struct = bpdb.PDBFile.read(io.StringIO(pdb_str)).get_structure(model=1)
ca = struct[struct.atom_name == "CA"]
coords0 = np.array(ca.coord, dtype=np.float64)  # (N, 3) equilibrium
N = len(coords0)
print(f"Cα count: {N}")

# ── Build full 3N×3N Hessian from ANM springs ──
from scipy.spatial.distance import pdist, squareform

CUTOFF = 13.0  # Å — standard ANM cutoff
K = 1.0  # uniform spring constant

dist_mat = squareform(pdist(coords0))  # (N, N)
adj = (dist_mat < CUTOFF) & (dist_mat > 0)

H = np.zeros((3 * N, 3 * N))
for i in range(N):
    for j in range(i + 1, N):
        if not adj[i, j]:
            continue
        r_vec = coords0[i] - coords0[j]  # (3,)
        r2 = r_vec @ r_vec
        block = -K * np.outer(r_vec, r_vec) / r2  # (3,3)
        H[3 * i : 3 * i + 3, 3 * j : 3 * j + 3] = block
        H[3 * j : 3 * j + 3, 3 * i : 3 * i + 3] = block
        H[3 * i : 3 * i + 3, 3 * i : 3 * i + 3] -= block
        H[3 * j : 3 * j + 3, 3 * j : 3 * j + 3] -= block

# ── Diagonalise ──
print("Diagonalising 3N×3N Hessian...", end=" ")
eigenvalues, eigenvectors = np.linalg.eigh(H)
print("done.")

# Sort by eigenvalue (should already be sorted, but enforce it)
idx = np.argsort(eigenvalues)
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

# Modes 0-5 are zero (rigid body). Mode 6 onwards = internal motions.
print(f"\nFirst 9 eigenvalues: {eigenvalues[:9].round(4)}")
print(f"Mode 7 (first internal, idx=6) eigenvalue: {eigenvalues[6]:.4f}")

## Mode 7: The Global Breathing Motion

Each eigenvector has 3N components — three displacement values (Δx, Δy, Δz) per Cα.
The **magnitude of displacement** at each residue tells us how much that residue moves
in this particular mode.

From all modes, we can compute the **theoretical B-factor** (crystallographic temperature
factor) and **RMSF** (root-mean-square fluctuation) — directly comparable to experimental
NMR order parameters S² (high RMSF → low S²).

In [ ]:
# Per-residue displacement in the top 5 non-trivial modes (6..10)
N_MODES_SHOWN = 5
mode_colors = ["#f72585", "#7209b7", "#3a0ca3", "#4361ee", "#4cc9f0"]
mode_labels = [f"Mode {i + 7}" for i in range(N_MODES_SHOWN)]

# RMSF from all non-trivial modes (equipartition: each mode contributes 1/λ)
kBT = 1.0  # in natural units
rmsf = np.zeros(N)
for k in range(6, 3 * N):  # skip rigid-body modes
    if eigenvalues[k] > 1e-6:
        mode_k = eigenvectors[:, k].reshape(N, 3)
        rmsf += kBT * np.sum(mode_k**2, axis=1) / eigenvalues[k]
rmsf = np.sqrt(rmsf)

res_ids = list(range(1, N + 1))

plt.style.use("seaborn-v0_8-darkgrid")
fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

# ── Panel 1: per-mode displacement profiles ──
ax1 = axes[0]
for i in range(N_MODES_SHOWN):
    mode_idx = 6 + i
    mode_vec = eigenvectors[:, mode_idx].reshape(N, 3)
    disp = np.linalg.norm(mode_vec, axis=1)
    disp_n = disp / disp.max()  # normalise each mode
    ax1.plot(res_ids, disp_n, color=mode_colors[i], lw=1.8, label=mode_labels[i], alpha=0.85)
    ax1.fill_between(res_ids, disp_n, alpha=0.08, color=mode_colors[i])

ax1.set_ylabel("Normalised Displacement", fontsize=11)
ax1.set_title(
    "Per-Residue Displacement in the 5 Lowest-Frequency Normal Modes\n"
    "Mode 7 = largest amplitude global motion",
    fontsize=12,
    fontweight="bold",
)
ax1.legend(fontsize=10, ncol=N_MODES_SHOWN)

# ── Panel 2: total RMSF ──
ax2 = axes[1]
norm_rmsf = plt.Normalize(rmsf.min(), rmsf.max())
colors_r = [cm.plasma(norm_rmsf(v)) for v in rmsf]
ax2.bar(res_ids, rmsf, color=colors_r, width=0.85, edgecolor="none")
ax2.axhline(rmsf.mean(), color="white", lw=1.5, ls="--", label=f"Mean RMSF = {rmsf.mean():.3f}")
sm = plt.cm.ScalarMappable(cmap="plasma", norm=norm_rmsf)
sm.set_array([])
plt.colorbar(sm, ax=ax2, label="RMSF", pad=0.01)
ax2.set_ylabel("RMSF (Å, ENM units)", fontsize=11)
ax2.set_xlabel("Residue Number", fontsize=11)
ax2.set_title(
    "Total RMSF from All Non-Trivial Modes (Equipartition)\n"
    "High RMSF → flexible | Low RMSF → rigid core",
    fontsize=12,
    fontweight="bold",
)
ax2.legend(fontsize=10)
ax2.set_xlim(1, N)

plt.tight_layout()
plt.savefig("normal_modes_rmsf.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Most flexible residue: {res_ids[np.argmax(rmsf)]} (RMSF={rmsf.max():.3f})")
print(f"Most rigid residue:    {res_ids[np.argmin(rmsf)]} (RMSF={rmsf.min():.3f})")

## Animating the Breathing Motion

We animate Mode 7 — the global lowest-frequency motion — by sinusoidally displacing
the Cα coordinates along the mode eigenvector. The amplitude is scaled to
±3 RMSF units to make the motion clearly visible.

Each residue is colored by its **displacement magnitude** in this mode:
🔴 bright red = large displacement (flexible ends/loops)
🔵 deep blue = minimal displacement (rigid core)

In [ ]:
# Animate Mode 7 as a 2D Cα trace (xy projection) — no 3D viewer needed
MODE = 6  # index into eigenvectors (0-indexed, so 6 = mode 7)
mode_vec = eigenvectors[:, MODE].reshape(N, 3)  # (N, 3)
disp_norms = np.linalg.norm(mode_vec, axis=1)  # per-residue displacement
AMPLITUDE = 3.0 * rmsf.mean() / disp_norms.max()  # scale to ~3 Å max

N_FRAMES = 40
thetas = np.linspace(0, 2 * np.pi, N_FRAMES, endpoint=False)

# Precompute all frames
anim_frames = []
for theta in thetas:
    displaced = coords0 + AMPLITUDE * np.sin(theta) * mode_vec
    anim_frames.append(displaced)

# Color by mode displacement magnitude (fixed across frames)
norm_d = plt.Normalize(disp_norms.min(), disp_norms.max())
colors_d = cm.RdYlBu_r(norm_d(disp_norms))  # red=flexible, blue=rigid

# Compute axis limits from all frames
all_xy = np.concatenate([f[:, :2] for f in anim_frames])
xmin, xmax = all_xy[:, 0].min() - 3, all_xy[:, 0].max() + 3
ymin, ymax = all_xy[:, 1].min() - 3, all_xy[:, 1].max() + 3

fig_a, ax_a = plt.subplots(figsize=(6.5, 6))
fig_a.patch.set_facecolor("#0d1117")
ax_a.set_facecolor("#0d1117")
for spine in ax_a.spines.values():
    spine.set_edgecolor("#313244")
ax_a.tick_params(colors="#8fa3bf")

(line_a,) = ax_a.plot([], [], "-", color="#444466", lw=1.2, zorder=1)
scat_a = ax_a.scatter(
    [], [], c=[], cmap="RdYlBu_r", s=55, vmin=disp_norms.min(), vmax=disp_norms.max(), zorder=2
)
sm_a = cm.ScalarMappable(cmap="RdYlBu_r", norm=norm_d)
sm_a.set_array([])
cbar_a = plt.colorbar(sm_a, ax=ax_a, fraction=0.046, pad=0.04)
cbar_a.set_label("Mode displacement\n(red=flexible, blue=rigid)", color="#cdd6f4", fontsize=9)
plt.setp(cbar_a.ax.yaxis.get_ticklabels(), color="#cdd6f4")
cbar_a.ax.yaxis.set_tick_params(color="#cdd6f4")

title_a = ax_a.set_title(
    "Mode 7 — Global Breathing of Ubiquitin (Frame 0)",
    color="#cdd6f4",
    fontsize=12,
    fontweight="bold",
)
ax_a.set_xlim(xmin, xmax)
ax_a.set_ylim(ymin, ymax)
ax_a.set_xlabel("x (Å)", color="#8fa3bf")
ax_a.set_ylabel("y (Å)", color="#8fa3bf")
plt.tight_layout()


def _init():
    line_a.set_data([], [])
    scat_a.set_offsets(np.empty((0, 2)))
    return line_a, scat_a, title_a


def _anim(fi):
    fr = anim_frames[fi]
    line_a.set_data(fr[:, 0], fr[:, 1])
    scat_a.set_offsets(fr[:, :2])
    scat_a.set_array(disp_norms)
    title_a.set_text(f"Mode 7 — Ubiquitin Breathing (frame {fi + 1}/{N_FRAMES})")
    return line_a, scat_a, title_a


ani_a = animation.FuncAnimation(
    fig_a, _anim, init_func=_init, frames=N_FRAMES, interval=60, blit=True
)
plt.close(fig_a)
HTML(ani_a.to_jshtml())

## Langevin Dynamics: Adding Thermal Noise

Pure normal modes give the *directions* of motion but assume **harmonic** behaviour and
ignore thermal noise. **Langevin dynamics** layers in two physical effects:

- **Friction** (`γ`): energy dissipation into the solvent (viscosity)
- **Random force** (`ξ`): thermal kicks from solvent molecules (the fluctuation-dissipation theorem)

$$m\ddot{x} = F_{ENM}(x) - \gamma \dot{x} + \xi(t), \quad \langle \xi(t)\xi(t') \rangle = 2k_BT\gamma\,\delta(t-t')$$

Running many steps of this integrator generates a **realistic conformational ensemble** —
the same kind of ensemble an NMR experiment samples over its measurement time.

In [ ]:
print("Running ENM Langevin dynamics (ubiquitin, 300 K)...")
with tempfile.NamedTemporaryFile(suffix=".pdb", delete=False, mode="w") as f:
    f.write(pdb_str)
    tmp_pdb = f.name

sys_d = System(tmp_pdb)
ff_d = ANMForceField(sys_d.equilibrium_coords, cutoff=13.0, spring_constant=0.8)
integrator = LangevinIntegrator(dt=0.05, temperature=300.0, friction=0.8)
sim_d = Simulation(sys_d, ff_d, integrator)

traj_path = tempfile.mktemp(suffix="_ubq_dyn.pdb")
N_STEPS = 500
STRIDE = 5  # 100 frames
sim_d.run(n_steps=N_STEPS, output_path=traj_path, stride=STRIDE)

# Read back trajectory
u_dyn = mda.Universe(traj_path)
traj_coords = []
for ts in u_dyn.trajectory:
    traj_coords.append(u_dyn.atoms.positions.copy())
traj_coords = np.array(traj_coords)  # (T, N, 3)
T = len(traj_coords)
print(f"Trajectory: {T} frames × {traj_coords.shape[1]} Cα atoms")

# Per-residue RMSF from trajectory
mean_pos = traj_coords.mean(axis=0)  # (N, 3)
rmsf_md = np.sqrt(((traj_coords - mean_pos[None]) ** 2).sum(axis=2).mean(axis=0))  # (N,)

os.unlink(tmp_pdb)
os.unlink(traj_path)

print("\nTrajectory RMSF summary:")
print(f"  Mean:  {rmsf_md.mean():.3f} Å")
print(f"  Max:   {rmsf_md.max():.3f} Å at residue {np.argmax(rmsf_md) + 1}")
print(f"  Min:   {rmsf_md.min():.3f} Å at residue {np.argmin(rmsf_md) + 1}")

## NMA vs Langevin: Two Views of the Same Flexibility

Both NMA and Langevin dynamics predict per-residue flexibility, but from different angles:

- **NMA RMSF** comes from the harmonic approximation — exact but ignores anharmonic motion
- **Langevin RMSF** includes thermal noise and anharmonic effects — more realistic but stochastic

The patterns should agree qualitatively: flexible termini and loops should show high RMSF
in both, while the structured core stays low. Discrepancies reveal anharmonic dynamics.

In [ ]:
fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

# Normalise both for comparison
rmsf_nma_n = rmsf / rmsf.max()
rmsf_md_n = rmsf_md / rmsf_md.max()

# Left: NMA RMSF
ax_l.fill_between(res_ids, rmsf_nma_n, alpha=0.25, color="#4cc9f0")
ax_l.plot(res_ids, rmsf_nma_n, color="#4cc9f0", lw=2.2)
ax_l.axhline(
    rmsf_nma_n.mean(),
    color="white",
    lw=1.2,
    ls="--",
    alpha=0.7,
    label=f"Mean = {rmsf_nma_n.mean():.2f}",
)
ax_l.set_title("NMA RMSF (Harmonic / Analytical)", fontsize=13, fontweight="bold")
ax_l.set_xlabel("Residue Number", fontsize=11)
ax_l.set_ylabel("Normalised RMSF", fontsize=11)
ax_l.legend(fontsize=10)
ax_l.set_xlim(1, N)

# Right: Langevin trajectory RMSF
ax_r.fill_between(res_ids, rmsf_md_n, alpha=0.25, color="#f72585")
ax_r.plot(res_ids, rmsf_md_n, color="#f72585", lw=2.2)
ax_r.axhline(
    rmsf_md_n.mean(),
    color="white",
    lw=1.2,
    ls="--",
    alpha=0.7,
    label=f"Mean = {rmsf_md_n.mean():.2f}",
)
ax_r.set_title(f"Langevin RMSF ({T} frames, 300 K)", fontsize=13, fontweight="bold")
ax_r.set_xlabel("Residue Number", fontsize=11)
ax_r.set_ylabel("Normalised RMSF", fontsize=11)
ax_r.legend(fontsize=10)
ax_r.set_xlim(1, N)

# Correlation
corr = np.corrcoef(rmsf_nma_n, rmsf_md_n)[0, 1]
fig.suptitle(f"NMA vs Langevin RMSF — Pearson r = {corr:.3f}", fontsize=14, fontweight="bold")
plt.style.use("seaborn-v0_8-darkgrid")
plt.tight_layout()
plt.savefig("rmsf_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"NMA vs Langevin RMSF correlation: r = {corr:.3f}")

## From Dynamics to Experiment: The NMR Connection

The RMSF profiles computed here connect directly to NMR observables:

| Computed quantity | NMR observable | How they relate |
|---|---|---|
| RMSF per residue | **S² order parameter** | High RMSF → low S² (flexible) |
| Mode 7 displacement | **hetNOE** | High displacement → low NOE |
| NMA eigenvalue λ | **τ_e internal correlation time** | τ_e ∝ 1/λ |

This means the animations you saw are not just pretty pictures — the **mode 7 breathing
motion** is the very motion that the **Lipari-Szabo S² parameter** measures. A residue
colored red in the animation has low S², high RMSF, and low hetNOE.

**Physiological relevance of ubiquitin's dynamics**:
- The flexible **C-terminal tail** (residues 73–76) is where E2 enzymes bind
- The **β-grasp fold** (residues 1–70) is the rigid signalling scaffold
- Mode 7 describes a twist/breathing that exposes the Ile44 hydrophobic patch —
  the primary recognition surface for most ubiquitin-binding domains

The protein's dynamics are not incidental — they are the mechanism of function.